In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# === CONFIG ===
API_URL = "https://api.weatherbit.io/v2.0/forecast/hourly"
API_KEY = "777628119ce049d484833355dbeca175"  # <-- Replace this
LAT = 28.6139   # Delhi
LON = 77.2090
HOURS = 48
OUTPUT_CSV = "dataset/delhi_weatherbit_inference_ready.csv"

# === FETCH ===
def fetch_weatherbit_data():
    params = {
        "lat": LAT,
        "lon": LON,
        "key": API_KEY,
        "hours": HOURS
    }

    r = requests.get(API_URL, params=params)
    r.raise_for_status()
    data = r.json()['data']

    df = pd.DataFrame(data)
    df['timestamp_local'] = pd.to_datetime(df['timestamp_local'])
    df.set_index('timestamp_local', inplace=True)

    df = compute_qv2m(df)
    df = rename_for_inference(df)
    print(df.head());

# === COMPUTE QV2M ===
def compute_qv2m(df):
    T = df['temp']
    RH = df['rh'] / 100.0
    P = df['pres']

    es = 6.112 * np.exp((17.67 * T) / (T + 243.5))
    e = RH * es
    qv = (0.622 * e) / (P - (1 - 0.622) * e)

    df['QV2M'] = qv
    return df

# === RENAME TO MODEL FORMAT ===
def rename_for_inference(df):
    renamed = df.rename(columns={
        "ghi": "ALLSKY_SFC_SW_DWN",
        "temp": "T2M"
    })

    return renamed[["ALLSKY_SFC_SW_DWN", "T2M", "QV2M"]]  # only keep needed

# === MAIN ===


In [3]:
fetch_weatherbit_data()

                     ALLSKY_SFC_SW_DWN   T2M      QV2M
timestamp_local                                       
2025-08-07 17:30:00                291  34.8  0.020503
2025-08-07 18:30:00                 78  34.1  0.020760
2025-08-07 19:30:00                  0  33.3  0.020822
2025-08-07 20:30:00                  0  32.5  0.020853
2025-08-07 21:30:00                  0  31.9  0.020745
